In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))


In [2]:
import gradio as gr

In [3]:
from Day4LLMCalling import Llms,mSeries

In [4]:
def laugh(text):
    print('laugh initiated...')
    return f'Laughing like a {text.upper()}' 
    

In [5]:
# laugh('wahahaha')

In [6]:
# gr.Interface(fn=Llms.callModel,inputs='textbox', outputs='textbox', flagging_mode='never').launch(inbrowser=True, auth=('edd','howdy'))

In [7]:
def wrapLlm(message,source):
    yield from Llms.callModelGenerator(message,source=source)

In [8]:
message_input = gr.Textbox(label="Your message:", info="Enter a message to be shouted", lines=7)
message_output = gr.Markdown(label="Response:")
message_model_selection = gr.Dropdown(choices=['ollama','gemini','openRouter'], label='Choose model')

view = gr.Interface(
    fn=wrapLlm,
    title="Shout", 
    inputs=[message_input,message_model_selection], 
    outputs=[message_output], 
    examples=[["hello there. Introduce yourself.",'openRouter'], ["howdy",'gemini']], 
    flagging_mode="never",
    theme='soft'
    )
view.launch()

c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [9]:
# Llms.callModel('hi there',stream=True,source='openRouter',model='openrouter/free')

In [10]:
def wrapLlm(message,source,model,temperature,chat_no):
    # if not source:
    #     source = 'openRouter'
    # if not model:
    #     model = 'gpt-oss:20b'
    yield from Llms.callModelGenerator(message,source=source, model=model, temperature=temperature, chat_no=chat_no)

In [17]:


def update_chatbox(model,chat_no):
    return mSeries.promptList.get(chat_no,{}).get(model,[])

def create_new_chat(chat_list):
    chat_list.append(f'Chat{len(chat_list)+1}')
    return chat_list, len(chat_list)

def load_chat(chat_num):
    return chat_num

with gr.Blocks(theme=gr.themes.Soft()) as demo:
        chat_list = gr.State(['Chat1']) 
        chat_no = gr.State(1)
        source = gr.State('openRouter')
        model_name = gr.State('openrouter/free')
        temperature = gr.State(0)
        
        with gr.Row():
            with gr.Column(scale=1, variant='panel'):

                new_chat_btn = gr.Button('New',variant='huggingface',size='sm')

                @gr.render(inputs=[chat_list])
                def render_chats(chat_list):
                    with gr.Group():
                        for i, chat in enumerate(chat_list) :
                            chat_num = gr.State(i+1)
                            btn = gr.Button(chat,size='lg', variant= 'stop')

                            btn.click(
                                fn=load_chat, 
                                inputs=[chat_num],
                                outputs=[chat_no]
                            ).then(
                                fn=update_chatbox, 
                                inputs=[model_name,chat_no], 
                                outputs=[chat_history] )

                new_chat_btn.click(
                    fn = create_new_chat, 
                    inputs=[chat_list], 
                    outputs=[chat_list,chat_no] )

                
            with gr.Column(scale=4):

                chat_history = gr.Chatbot(label='Chat History')    
                response_box = gr.Markdown(label='Last response')
                    
                with gr.Group():    
                    user_input = gr.Textbox(
                        placeholder='Enter your prompt',
                        show_label=False,
                        scale=8)
                    submit_btn= gr.Button(
                        'enter',
                        size='sm',
                        variant='primary',
                        scale=1)

                submit_btn.click(
                    wrapLlm,
                    inputs=[user_input,source,model_name,temperature,chat_no], 
                    outputs=[response_box]
                ).then(
                    fn=update_chatbox, 
                    inputs=[model_name,chat_no], 
                    outputs=chat_history
                ).then(
                    fn = lambda rst='':rst, 
                    outputs=[user_input] )

                user_input.submit(
                    wrapLlm,
                    inputs=[user_input,source,model_name,temperature,chat_no], 
                    outputs=[response_box]
                ).then(
                    fn=update_chatbox, 
                    inputs=[model_name,chat_no], 
                    outputs=chat_history
                ).then(
                    fn = lambda rst='':rst,
                    outputs=[user_input])


            with gr.Column(scale=1):
                with gr.Accordion('Adv_settings'):
                    source_selection = gr.Dropdown(
                        choices=['openRouter','gemini','ollama'],
                        label='Select source-')

                    source_selection.change(
                        fn= lambda source:source, 
                        inputs=[source_selection],
                        outputs= [source])

                    temperature_select = gr.Slider(
                        0,2,value=0,
                        step=0.1,
                        label='temp_slider') 

                    model_name_input = gr.Textbox(
                        placeholder='Enter model name',
                        value='openrouter/free',
                        label='Model name')
                    
                    model_name_input.submit(
                        fn = lambda model_name:model_name,
                        inputs=[model_name_input],
                        outputs=[model_name])

                    temperature_select.change(
                        fn = lambda temperature:temperature, 
                        inputs=[temperature_select],
                        outputs=[temperature])

            
                files = gr.File(label='insert file',file_count='single',file_types=['pdf'])  

demo.launch()

C:\Users\PGCP-AI\AppData\Local\Temp\ipykernel_19324\2211834877.py:11: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [12]:
mSeries.promptList

{}

In [13]:
import gradio as gr

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    with gr.Row():
        # LEFT COLUMN: Sidebar (Scale 1)
        with gr.Column(scale=1, variant="panel"):
            gr.Markdown("### 💬 Chats")
            new_chat_btn = gr.Button('＋ New Chat', variant='primary', size='sm')
            # Using buttons for a cleaner sidebar look
            gr.Button("History: Project Alpha", size="sm", variant="secondary")
            gr.Button("History: Code Review", size="sm", variant="secondary")

        # MIDDLE COLUMN: Main Chat (Scale 4)
        with gr.Column(scale=4):
            chat_history = gr.Chatbot(height=500)    
            with gr.Group(): # Bonds textbox and button together
                user_input = gr.Textbox(
                    placeholder='Enter your prompt...',
                    show_label=False,
                    container=False
                )
                submit_btn = gr.Button("Send", variant="primary")

        # RIGHT COLUMN: Settings (Scale 1.5)
        with gr.Column(scale=1.5, variant="panel"):
            gr.Markdown("### ⚙️ Configuration")
            source_selection = gr.Dropdown(
                choices=['ollama', 'gemini', 'openRouter'], 
                label='Model Provider', 
                value='ollama'
            )
            
            with gr.Accordion('Model Parameters', open=True):
                temperature = gr.Slider(0, 100, value=70, label='Creativity (Temp)')
                top_p = gr.Slider(0, 1, value=0.9, label='Top P')
            
            files = gr.File(
                label='Reference PDF', 
                file_count='single', 
                file_types=['.pdf']
            )

demo.launch()

C:\Users\PGCP-AI\AppData\Local\Temp\ipykernel_19324\3522000354.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\layouts\column.py:59: UserWarning: 'scale' value should be an integer. Using 1.5 will cause issues.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
